In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Read sampled dataset with Amazon inventory data

In [3]:
df_items=pd.read_json('../../data/meta_Electronics_2022_2023_has_main_category_sample_1000.jsonl', lines=True)

In [4]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Camera & Photo,GuFamily Indoor Security Camera 2K HD 360 Degr...,1.7,8,[【2K Clarity for Full Coverage Monitoring】: Th...,[2K HD Indoor Surveillance Camera Works with 2...,18.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],generic,"[Electronics, Camera & Photo, Video Surveillan...",{'Package Dimensions': '6.18 x 3.54 x 3.43 inc...,B0CBNZ93NJ,NaN,NaN,NaN
1,Cell Phones & Accessories,Tinkers Silicone Airpod Pro Case with Airtag H...,4.2,9,[Airpod Pro Keyring Case with Airtag Holder Fo...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '2 in 1 Silicone Protective Case fo...,Tinkers,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '3.86 x 3.62 x 1.3 inch...,B09Q5HMPLS,NaN,NaN,NaN
2,All Electronics,Cameron sino 2200mAh 3.7V Li-ion 361-00023-13 ...,3.0,1,[Replacement for models: Garmin Pro 550 handhe...,[Cameron Sino is a brand focusing on sell batt...,25.78,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cameron Sino®,[],{'Product Dimensions': '2.61 x 0.73 x 0.8 inch...,B01LZ85O5N,NaN,NaN,NaN
3,Computers,LTPRPTS Replacement Laptop LCD Back Cover Top ...,5.0,5,[1. 100% Original Brand New for Lenovo Flex 5 ...,[New Replacement for Lenovo Flex 5 15 5 15iil0...,55.68,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Installation video ONLY for IdeaPa...,LTPRPTS,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '13.78 x 9.84 x 3.94 in...,B0B7X9CQTF,NaN,NaN,NaN
4,Camera & Photo,ETLIN Security Camera Indoor with Smoke Detect...,4.5,112,[Security Camera Smoke Detector: 2-in-1 design...,"[Home Camera Featuring:, ●HD 1080P Video ●HD 1...",45.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],ETLIN,"[Electronics, Camera & Photo, Video Surveillan...","{'Package Dimensions': '3 x 1.1 x 1.1 inches',...",B0C6GCBXMT,NaN,NaN,NaN


In [5]:
list(df_items["features"].items())[0]

(0,
 ['【2K Clarity for Full Coverage Monitoring】: This indoor security camera boasts a 2K resolution, combined with 360-degree horizontal rotation and 105-degree vertical tilt, allowing you to effectively monitor your home. The full coverage monitoring of this home camera is not only for space,but also for time, the night vision function provides you with 24-hour continuous monitoring without interruption, so you can know what happened at night.',
  '【2.4GHz&5GHzDual-Band WiFi Connection】: This home camera works with 2.4GHz & 5GHz WiFi, and easy to install, even can not install on the wall, you can just put it on the shelf or any flat area. It can be installed on the ceiling if needed. So you neither need to worry about whether this camera can connect to your network, nor distressed about how to install this camera.',
  '【Motion Detection & Tracking Alert】: Indoor security cameras for home automatically track motion when detected. When motion is sensed, the App will promptly send a rea

In [6]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/41kLHj6mnHL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41kLHj6mnHL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/61nfN2SWotL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51EFWksD7QL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51EFWksD7QL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71dp-ztU1DL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51U87TVdKmL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51U87TVdKmL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71xkh9zbuPL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51F85a5mdkL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51F85a5mdkL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71N4+MeBiNL._AC_SL1

In [7]:
def preprocess_description(row):
    return f"{row["title"]} {' '.join(row["features"])}"

In [8]:
def extract_first_large_image(row):
    try:
        return row["images"][0].get("large", "")
    except (IndexError, TypeError, KeyError):
        return ""

In [9]:
df_items["description"] = df_items.apply(preprocess_description,axis=1)
df_items["image"]=df_items.apply(extract_first_large_image,axis=1)

In [10]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Camera & Photo,GuFamily Indoor Security Camera 2K HD 360 Degr...,1.7,8,[【2K Clarity for Full Coverage Monitoring】: Th...,GuFamily Indoor Security Camera 2K HD 360 Degr...,18.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],generic,"[Electronics, Camera & Photo, Video Surveillan...",{'Package Dimensions': '6.18 x 3.54 x 3.43 inc...,B0CBNZ93NJ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41kLHj6mnH...
1,Cell Phones & Accessories,Tinkers Silicone Airpod Pro Case with Airtag H...,4.2,9,[Airpod Pro Keyring Case with Airtag Holder Fo...,Tinkers Silicone Airpod Pro Case with Airtag H...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '2 in 1 Silicone Protective Case fo...,Tinkers,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '3.86 x 3.62 x 1.3 inch...,B09Q5HMPLS,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31J5BQZchC...
2,All Electronics,Cameron sino 2200mAh 3.7V Li-ion 361-00023-13 ...,3.0,1,[Replacement for models: Garmin Pro 550 handhe...,Cameron sino 2200mAh 3.7V Li-ion 361-00023-13 ...,25.78,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cameron Sino®,[],{'Product Dimensions': '2.61 x 0.73 x 0.8 inch...,B01LZ85O5N,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41tSkDcUta...
3,Computers,LTPRPTS Replacement Laptop LCD Back Cover Top ...,5.0,5,[1. 100% Original Brand New for Lenovo Flex 5 ...,LTPRPTS Replacement Laptop LCD Back Cover Top ...,55.68,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Installation video ONLY for IdeaPa...,LTPRPTS,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '13.78 x 9.84 x 3.94 in...,B0B7X9CQTF,NaN,NaN,NaN,https://m.media-amazon.com/images/I/21+U-0xlpg...
4,Camera & Photo,ETLIN Security Camera Indoor with Smoke Detect...,4.5,112,[Security Camera Smoke Detector: 2-in-1 design...,ETLIN Security Camera Indoor with Smoke Detect...,45.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],ETLIN,"[Electronics, Camera & Photo, Video Surveillan...","{'Package Dimensions': '3 x 1.1 x 1.1 inches',...",B0C6GCBXMT,NaN,NaN,NaN,https://m.media-amazon.com/images/I/314Icv83q2...


In [11]:
df_items["description"]

0      GuFamily Indoor Security Camera 2K HD 360 Degr...
1      Tinkers Silicone Airpod Pro Case with Airtag H...
2      Cameron sino 2200mAh 3.7V Li-ion 361-00023-13 ...
3      LTPRPTS Replacement Laptop LCD Back Cover Top ...
4      ETLIN Security Camera Indoor with Smoke Detect...
                             ...                        
995    Zip Ties Heavy Duty(150Pcs), Cable Ties Reusab...
996    MOROVOL ATX PC Case Pre-Installed 4PCS RGB Fan...
997    Smart Watch for Women Ladies Smart Bracelet IP...
998    NTONPOWER 2 Prong Power Strip, 1875W 2 Prong t...
999    GAROGYI Bluetooth 5.1 Adapter for PC, USB Blue...
Name: description, Length: 1000, dtype: object

### Sample 50 items from dataset

In [12]:
df_sample = df_items.sample(50,random_state=42)

In [13]:
len(df_sample)

50

In [14]:
data_to_embed = df_sample[["description","image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [15]:
data_to_embed

[{'description': '2023 Upgraded 1080P WiFi Camera Motion Sound Detector for Home Office,Indoor Camera with Night Vision - Car Cameras for Surveillance - Size:1.22 x 1.22 x 1.22 inches ',
  'image': 'https://m.media-amazon.com/images/I/41wwiQiqHPL._AC_.jpg',
  'rating_number': 19,
  'price': 18.85,
  'average_rating': 2.1,
  'parent_asin': 'B0BZYSHG74'},
 {'description': '65W Replacement AC Adapter Charger for HP 15 15-f233wm 15-f272wm 15-f010wm 15-f305dx 15-af131dx Spectre X360 13-a010dx 13-4003dx 13-4103dx Pavilion 15 Laptop Notebook PC Power Supply Cord FEATURES / POWER SPECS : Only Pwr+ Chargers Have Extra Long 12 Ft Power AC/DC cords / Output 19.5V 3.33A 45W 65W / Input 100-240V / Made in Taiwan COMPATIBILITY: HP 14 15 17 M6 M7 Series 14-dq0040nr 463958-001 741727-001 740015-001 740015-003 740015-004 Envy Stream 11 13 14 PRO G2 HP Spectre x360 13 13T Split x2, 612 G1, 410 G1, Elite X2 1011 G1, x360 310 G2, ZBook 14 HP Chromebook 11 14 G1 G3 G4 Pavilion x360 14-dw1024nr; HP Notebook

###Embedding function definition

In [16]:
response = openai.embeddings.create(
    input="Random text",
    model="text-embedding-3-small",
)

In [17]:
len(response.data[0].embedding)

1536

In [18]:
def get_embedding(text,model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [19]:
get_embedding("Hi")

[-0.006954193115234375,
 -0.03533935546875,
 0.0015869140625,
 0.06536865234375,
 0.032928466796875,
 -0.0242156982421875,
 -0.026123046875,
 0.049407958984375,
 0.0162506103515625,
 -0.05169677734375,
 -0.01343536376953125,
 -0.0144805908203125,
 -0.0260009765625,
 -0.0032596588134765625,
 0.02459716796875,
 0.0011425018310546875,
 -0.053497314453125,
 0.01508331298828125,
 0.01148223876953125,
 0.033935546875,
 0.049285888671875,
 0.020355224609375,
 -0.01395416259765625,
 0.01885986328125,
 0.0171661376953125,
 0.0241851806640625,
 0.0182647705078125,
 -0.0012063980102539062,
 0.0196075439453125,
 -0.03680419921875,
 0.027679443359375,
 -0.0282440185546875,
 0.0276641845703125,
 -0.0162200927734375,
 -0.0117645263671875,
 -0.016021728515625,
 -0.0140838623046875,
 0.037567138671875,
 0.0189208984375,
 -0.03765869140625,
 0.04345703125,
 -0.012420654296875,
 0.0209503173828125,
 0.0135040283203125,
 0.0019893646240234375,
 0.00025463104248046875,
 -0.04913330078125,
 0.00832366943359

### Quadrant Collection

In [20]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [22]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `Amazon-items-collection-00` already exists!"},"time":0.719994993}'

### Embed Data

##### Test

In [23]:
pointstruct= PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text":"Test text",
        "model":"text-embedding-3-small",
    },
    )

In [24]:
pointstruct

PointStruct(id=0, vector=[-0.020111083984375, 0.0070037841796875, 0.0377197265625, -0.040252685546875, -0.0191802978515625, -0.0343017578125, 0.000576019287109375, -0.024261474609375, 0.038665771484375, 0.0015888214111328125, 0.0306243896484375, 0.01277923583984375, -0.01087188720703125, 0.011260986328125, 0.026397705078125, 0.04302978515625, -0.04681396484375, -0.006893157958984375, -0.021728515625, 0.05108642578125, 0.007061004638671875, 0.01308441162109375, 0.01129150390625, -0.032501220703125, -0.003940582275390625, -0.0390625, -0.03692626953125, 0.0050048828125, 0.058441162109375, -0.07452392578125, 0.0313720703125, -0.0440673828125, -0.0002703666687011719, -0.010955810546875, 0.00604248046875, 0.0382080078125, 0.0303192138671875, 0.03759765625, 0.0102386474609375, -0.02935791015625, -0.014923095703125, -0.01042938232421875, 0.021270751953125, 0.0181427001953125, 0.02667236328125, 0.00891876220703125, -0.01177978515625, 0.00879669189453125, 0.01995849609375, 0.037933349609375, -0.

### Amazon data

In [25]:
poinstructs = []
for i, data in enumerate(data_to_embed):
    pointstruct = PointStruct(
        id=i,
        vector=get_embedding(data["description"]),
        payload=data,
    )
    poinstructs.append(pointstruct)

In [26]:
poinstructs

[PointStruct(id=0, vector=[0.0207977294921875, 0.001369476318359375, -0.017486572265625, 0.0015239715576171875, -0.0416259765625, -0.020050048828125, 0.025634765625, -0.0034580230712890625, 0.003925323486328125, -0.0195159912109375, 0.0098114013671875, 0.042388916015625, -0.051025390625, -0.0164337158203125, 0.0290679931640625, -0.0012540817260742188, -0.01528167724609375, 0.015380859375, -0.0183258056640625, -0.033477783203125, 0.001750946044921875, 0.038330078125, 0.01427459716796875, -0.04010009765625, 0.06414794921875, -0.007572174072265625, -0.007106781005859375, -0.01187896728515625, 0.0380859375, 0.02215576171875, -0.00417327880859375, -0.0528564453125, -0.0287322998046875, -0.0489501953125, -0.021148681640625, -0.0008411407470703125, -0.05029296875, -0.046661376953125, -0.0186767578125, 0.00664520263671875, 0.034027099609375, 0.042755126953125, 0.00943756103515625, 0.00830841064453125, -0.0279388427734375, -0.01495361328125, -0.06787109375, 0.0350341796875, -0.06658935546875, -

In [27]:
len(poinstructs)

50

### Write embedded data to qdrant

In [28]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    points=poinstructs,
    wait=True,
)

ResponseHandlingException: timed out

### Data retrieval function

In [32]:
def retrieve_data(query, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )
    return search_result

### Test Retrieval

In [34]:
retrieve_data("What kind of charging cords do you offer?",top_k=10).points

[ScoredPoint(id=10, version=1, score=0.37975973, payload={'description': 'Bauhoo Charging Station for Apple Multiple Devices, 3 in 1 Fast Wireless Charger Foldable iPhone 14/13/12/11/Pro/XS/Xs Max/XR/X/SE/8/8 Plus Watch 8/7/6/SE/5/4/3/2 AirPods 3/2/Pro with Adapter Pink 0', 'image': 'https://m.media-amazon.com/images/I/41iK0-+HVSL._AC_.jpg', 'rating_number': 159, 'price': 17.75, 'average_rating': 4.1, 'parent_asin': 'B0BHRHS1Q7'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=1, score=0.36270505, payload={'description': '65W Replacement AC Adapter Charger for HP 15 15-f233wm 15-f272wm 15-f010wm 15-f305dx 15-af131dx Spectre X360 13-a010dx 13-4003dx 13-4103dx Pavilion 15 Laptop Notebook PC Power Supply Cord FEATURES / POWER SPECS : Only Pwr+ Chargers Have Extra Long 12 Ft Power AC/DC cords / Output 19.5V 3.33A 45W 65W / Input 100-240V / Made in Taiwan COMPATIBILITY: HP 14 15 17 M6 M7 Series 14-dq0040nr 463958-001 741727-001 740015-001 740015-003 740015-004 En